In [70]:
from IPython.display import Image
import pandas as pd
import numpy as np
from pandas.conftest import axis_1
from scipy.signal import square

In [71]:
data = pd.read_csv('ks.csv')

In [72]:
data.head()

,Название,Категория,Главная категория,Валюта,Дедлайн,Дата публикации,Состояние,Инвесторов,Страна,Собрано в долларах,Цель в долларах
0,"Don't Call it a Comeback ""Telescopes""",Music,Music,USD,2013-01-10,2012-12-09 06:03:52,successful,23,US,600.00,600.00
1,Arcade County (Canceled),Games,Games,USD,2012-04-29,2012-03-30 23:40:45,canceled,5,US,71.00,9000.00
2,Hayashi Skate Co. Solar Skateboard backpack,Accessories,Fashion,CAD,2017-07-22,2017-05-23 23:00:13,canceled,8,CA,360.36,2391.77
3,Me & You Coordinating Sunglasses- Optical Qual...,Accessories,Fashion,USD,2016-11-18,2016-10-19 22:06:41,failed,20,US,502.00,10000.00
4,New Carts for Istanbul Street Food Vendors,Food,Food,USD,2015-05-17,2015-04-17 18:10:47,successful,62,US,2414.00,1400.00


In [73]:
data.shape

(378661, 11)

In [74]:
data['Состояние'].value_counts()

Состояние
failed        197719
successful    133956
canceled       38779
undefined       3562
live            2799
suspended       1846
Name: count, dtype: int64

In [75]:
data = data[data['Состояние'].isin(['failed', 'successful'])]
data['Состояние'].value_counts()

Состояние
failed        197719
successful    133956
Name: count, dtype: int64

## Задача сводиться к классификации и регрессии
### Что берем за таргет? 2 варианта:

- Будем решать задачу классификации, разметив объекты следующим образом: те проекты, у которых успешный статус, единичкой, а остальные ноликами

- Можем предсказывать просто собранное количество денег, применять модель, а потом уже смотреть, нужная ли сумма получилось. Тогда мы решаем задачу регрессии.


In [76]:
data.loc[(data['Состояние'] == 'failed'), 'target'] = 0
data.loc[(data['Состояние'] == 'successful'), 'target'] = 1
data.head()

,Название,Категория,Главная категория,Валюта,Дедлайн,Дата публикации,Состояние,Инвесторов,Страна,Собрано в долларах,Цель в долларах,target
0,"Don't Call it a Comeback ""Telescopes""",Music,Music,USD,2013-01-10,2012-12-09 06:03:52,successful,23,US,600.00,600.0,1.0
3,Me & You Coordinating Sunglasses- Optical Qual...,Accessories,Fashion,USD,2016-11-18,2016-10-19 22:06:41,failed,20,US,502.00,10000.0,0.0
4,New Carts for Istanbul Street Food Vendors,Food,Food,USD,2015-05-17,2015-04-17 18:10:47,successful,62,US,2414.00,1400.0,1.0
5,New Improv Comedy Venue in Des Moines,Theater,Theater,USD,2013-06-17,2013-05-03 16:17:21,successful,86,US,10030.88,10000.0,1.0
6,The Seer and the Sword,Shorts,Film & Video,USD,2012-08-11,2012-07-12 05:19:53,failed,0,US,0.00,10000.0,0.0


In [77]:
data = data.drop('Состояние', axis=1)

## Регрессия

In [78]:
data = data.rename({'Собрано в долларах':'target2'}, axis=1)
data.head()

,Название,Категория,Главная категория,Валюта,Дедлайн,Дата публикации,Инвесторов,Страна,target2,Цель в долларах,target
0,"Don't Call it a Comeback ""Telescopes""",Music,Music,USD,2013-01-10,2012-12-09 06:03:52,23,US,600.00,600.0,1.0
3,Me & You Coordinating Sunglasses- Optical Qual...,Accessories,Fashion,USD,2016-11-18,2016-10-19 22:06:41,20,US,502.00,10000.0,0.0
4,New Carts for Istanbul Street Food Vendors,Food,Food,USD,2015-05-17,2015-04-17 18:10:47,62,US,2414.00,1400.0,1.0
5,New Improv Comedy Venue in Des Moines,Theater,Theater,USD,2013-06-17,2013-05-03 16:17:21,86,US,10030.88,10000.0,1.0
6,The Seer and the Sword,Shorts,Film & Video,USD,2012-08-11,2012-07-12 05:19:53,0,US,0.00,10000.0,0.0


In [79]:
data['Дедлайн'] = pd.to_datetime(data['Дедлайн'])
data['Дата публикации'] = pd.to_datetime(data['Дата публикации'])

In [80]:
data['Срок'] = (data['Дедлайн'] - data['Дата публикации']).dt.days


In [81]:
data['Год публикации'] = data['Дата публикации'].dt.year

In [82]:
data.head()

,Название,Категория,Главная категория,Валюта,Дедлайн,Дата публикации,Инвесторов,Страна,target2,Цель в долларах,target,Срок,Год публикации
0,"Don't Call it a Comeback ""Telescopes""",Music,Music,USD,2013-01-10,2012-12-09 06:03:52,23,US,600.00,600.0,1.0,31,2012
3,Me & You Coordinating Sunglasses- Optical Qual...,Accessories,Fashion,USD,2016-11-18,2016-10-19 22:06:41,20,US,502.00,10000.0,0.0,29,2016
4,New Carts for Istanbul Street Food Vendors,Food,Food,USD,2015-05-17,2015-04-17 18:10:47,62,US,2414.00,1400.0,1.0,29,2015
5,New Improv Comedy Venue in Des Moines,Theater,Theater,USD,2013-06-17,2013-05-03 16:17:21,86,US,10030.88,10000.0,1.0,44,2013
6,The Seer and the Sword,Shorts,Film & Video,USD,2012-08-11,2012-07-12 05:19:53,0,US,0.00,10000.0,0.0,29,2012


### Чтобы получить матрицу объектов, зачастую нужно обработать сырые данные, то есть извлечь из имеющихся таблиц признаки там, где они не даны явно

In [83]:
macro = pd.read_excel('macrofeatures.xlsx', engine='openpyxl')
macro.head()

,Unnamed: 0,Close_brent,Close_sugar,Close_cereals,Close_index_moex,Close_index_moex_10,Close_index_RGBI,Close_index_RTS_oil_and_gas,Close_index_RTS_metallurgy,Close_index_RTS_consumer_sector,Close_index_RTS_telecom,Close_index_RTS_finance,Close_index_RTS_transport,Close_index_RTS_chemicals,Close_index_RTS_broad_market,Close_index_RTS_electricity,dlk_cob_date
0,0,34.41,13.97,442.75,1797.27,3940.81,125.59,123.40,111.97,196.55,70.17,140.57,27.06,177.38,530.59,32.49,2016-02-24
1,1,35.06,14.24,445.25,1803.89,3977.35,126.44,124.22,112.51,198.03,70.56,142.64,27.43,179.48,536.20,33.07,2016-02-25
2,2,35.13,14.00,443.25,1816.73,4027.23,126.90,125.38,113.44,200.13,71.94,145.45,28.06,181.56,544.73,33.55,2016-02-26
3,3,36.64,14.36,445.00,1840.17,4084.24,126.87,126.69,114.66,200.32,72.41,147.22,28.49,186.76,552.82,34.41,2016-02-29
4,4,36.60,14.39,438.50,1844.17,4087.06,127.78,129.72,117.09,204.30,74.26,150.04,30.12,190.67,565.45,34.96,2016-03-01


In [84]:
macro = macro[['Close_brent', 'dlk_cob_date']].drop_duplicates()
macro['dlk_cob_date'] = pd.to_datetime(macro['dlk_cob_date'])

data.head()
data = pd.merge(data, macro,
         left_on=['Дата публикации'],
         right_on=['dlk_cob_date'],
         how='left')

In [85]:
data = data.sort_values('Дата публикации')
data.head()

,Название,Категория,Главная категория,Валюта,Дедлайн,Дата публикации,Инвесторов,Страна,target2,Цель в долларах,target,Срок,Год публикации,Close_brent,dlk_cob_date
176128,Grace Jones Does Not Give A F$#% T-Shirt (limi...,Fashion,Fashion,USD,2009-05-31,2009-04-21 21:02:48,30,US,625.0,1000.0,0.0,39,2009,NaN,NaT
241929,CRYSTAL ANTLERS UNTITLED MOVIE,Shorts,Film & Video,USD,2009-07-20,2009-04-23 00:07:53,3,US,22.0,80000.0,0.0,87,2009,NaN,NaT
244460,drawing for dollars,Illustration,Art,USD,2009-05-03,2009-04-24 21:52:03,3,US,35.0,20.0,1.0,8,2009,NaN,NaT
80845,Offline Wikipedia iPhone app,Software,Technology,USD,2009-07-14,2009-04-25 17:36:21,25,US,145.0,99.0,1.0,79,2009,NaN,NaT
181197,Pantshirts,Fashion,Fashion,USD,2009-05-26,2009-04-27 14:10:39,10,US,387.0,1900.0,0.0,28,2009,NaN,NaT


In [86]:
data['Close_brent'] = data['Close_brent'].fillna(data['Close_brent'].mean())

In [87]:
data = data.drop(['Дедлайн', 'Дата публикации', 'dlk_cob_date'], axis=1)


In [88]:
data = data.drop(['Название', 'Страна', 'Инвесторов'], axis=1)

In [89]:
data = pd.concat((data, pd.get_dummies(data['Валюта'])), axis=1)
data = data.drop(['Валюта'], axis=1)

In [90]:
data = data.drop(['AUD'], axis=1)

In [91]:
data = pd.concat((data, pd.get_dummies(data['Главная категория'])), axis=1)
data = data.drop(['Главная категория'], axis=1)

In [92]:
data = data.drop(['Games'], axis=1)

In [93]:
data['Категория'] = data['Категория'].map(data.groupby(['Категория'])['target2'].mean())

In [94]:
data.head()

,Категория,target2,Цель в долларах,target,Срок,Год публикации,Close_brent,CAD,CHF,DKK,...,Design,Fashion,Film & Video,Food,Journalism,Music,Photography,Publishing,Technology,Theater
176128,6035.989239,625.0,1000.0,0.0,39,2009,48.505,False,False,False,...,False,True,False,False,False,False,False,False,False,False
241929,3591.033473,22.0,80000.0,0.0,87,2009,48.505,False,False,False,...,False,False,True,False,False,False,False,False,False,False
244460,3661.424550,35.0,20.0,1.0,8,2009,48.505,False,False,False,...,False,False,False,False,False,False,False,False,False,False
80845,4321.245721,145.0,99.0,1.0,79,2009,48.505,False,False,False,...,False,False,False,False,False,False,False,False,True,False
181197,6035.989239,387.0,1900.0,0.0,28,2009,48.505,False,False,False,...,False,True,False,False,False,False,False,False,False,False


## Определимся с таргетом

In [95]:
X = data.drop(['target2', 'target'], axis=1)
Y = data['target2']


### sklearn

In [96]:
from sklearn.linear_model import LinearRegression

In [97]:
model = LinearRegression()
model.fit(X, Y)
X['Предсказание'] = model.predict(X)
X.head()

,Категория,Цель в долларах,Срок,Год публикации,Close_brent,CAD,CHF,DKK,EUR,GBP,...,Fashion,Film & Video,Food,Journalism,Music,Photography,Publishing,Technology,Theater,Предсказание
176128,6035.989239,1000.0,39,2009,48.505,False,False,False,False,False,...,True,False,False,False,False,False,False,False,False,3125.878031
241929,3591.033473,80000.0,87,2009,48.505,False,False,False,False,False,...,False,True,False,False,False,False,False,False,False,5117.839135
244460,3661.424550,20.0,8,2009,48.505,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,-1680.562044
80845,4321.245721,99.0,79,2009,48.505,False,False,False,False,False,...,False,False,False,False,False,False,False,True,False,4935.969700
181197,6035.989239,1900.0,28,2009,48.505,False,False,False,False,False,...,True,False,False,False,False,False,False,False,False,2183.766185


## Оценка модели

### MSE

In [102]:
(((X['Предсказание'] - Y)**2).mean())**(1/2)

np.float64(95926.36316736735)

### MAE

In [104]:
abs(X['Предсказание'] - Y).mean()

np.float64(13854.013934531977)